# 01 Project Orientation

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roymustang11/InSAR-Benchmark-Lab/blob/main/notebooks/01_project_orientation.ipynb)

This notebook defines the first flagship case study for **InSAR Benchmark Lab**: groundwater-related land subsidence in California's Central Valley.

The purpose is not to produce a deformation result yet. The purpose is to make the scientific target, study area, data sources, and validation workflow explicit before processing starts.


## Scientific Question

> How reliably can open InSAR products measure groundwater-related land subsidence in California's Central Valley when checked against independent GNSS observations?

The first validation target is not only whether the deformation map looks plausible. The benchmark needs to quantify:

- agreement between InSAR and GNSS displacement time series,
- velocity differences in mm/year,
- sensitivity to reference-area selection,
- sensitivity to masking and coherence thresholds,
- whether uncertainty estimates are calibrated against observed residuals.


## Setup

This cell works in two modes:

- **Local repo mode:** run the notebook from this repository.
- **Colab mode:** clone the GitHub repository into `/content` and import the local package from `src/`.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/roymustang11/InSAR-Benchmark-Lab.git"

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    project_root = Path("/content/InSAR-Benchmark-Lab")
    if not project_root.exists():
        subprocess.check_call(["git", "clone", REPO_URL, str(project_root)])
else:
    cwd = Path.cwd().resolve()
    project_root = cwd.parent if cwd.name == "notebooks" else cwd

src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

config_path = project_root / "configs" / "central_valley_subsidence.yml"
print(f"Project root: {project_root}")
print(f"Config path: {config_path}")


## Study-Area Configuration

The study-area bounds and data-source assumptions live in `configs/central_valley_subsidence.yml` so future notebooks reuse the same target definition.


In [ ]:
from insar_benchmark_lab.config import load_study_area_config

config = load_study_area_config(config_path)
config


In [ ]:
bbox = config.region

print(f"Study area: {config.name}")
print(f"Application: {config.application}")
print(f"Time window: {config.time_window['start']} to {config.time_window['end']}")
print("Bounding box:")
print(f"  west/east: {bbox['west']} to {bbox['east']}")
print(f"  south/north: {bbox['south']} to {bbox['north']}")
print("InSAR sources:", ", ".join(config.data_sources["insar"]))
print("GNSS sources:", ", ".join(config.data_sources["gnss"]))


## Study-Area Map

This is a lightweight orientation map using only Matplotlib. It intentionally avoids external basemap downloads so the first notebook remains robust in local and Colab environments.


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

west, south, east, north = bbox["west"], bbox["south"], bbox["east"], bbox["north"]

fig, ax = plt.subplots(figsize=(7, 6))
ax.set_title("Central Valley Subsidence Study Area", fontsize=14)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_xlim(-125.0, -114.0)
ax.set_ylim(32.0, 42.5)
ax.grid(True, linestyle="--", linewidth=0.6, alpha=0.5)

study_box = Rectangle(
    (west, south),
    east - west,
    north - south,
    facecolor="#d55e00",
    edgecolor="#8b2d00",
    linewidth=2,
    alpha=0.25,
    label="Initial benchmark box",
)
ax.add_patch(study_box)
ax.text(west, north + 0.25, "Central Valley\ninitial benchmark", ha="left", va="bottom")

ax.scatter([-119.8], [36.3], marker="o", color="#0072b2", label="Approx. Central Valley context")
ax.legend(loc="lower left")
plt.show()


## Candidate Open Data Sources

| Source | Role in benchmark | Link |
| --- | --- | --- |
| OPERA DISP-S1 | Analysis-ready Sentinel-1 surface displacement products for open validation experiments. | [NASA Earthdata OPERA project](https://www.earthdata.nasa.gov/data/projects/opera), [OPERA products docs](https://nasa-opera.github.io/docs/products/) |
| ASF HyP3 | On-demand Sentinel-1 InSAR products and product packages. | [ASF HyP3 InSAR product guide](https://hyp3-docs.asf.alaska.edu/hyp3-docs/guides/insar_product_guide/) |
| ARIA GUNW | Geocoded unwrapped interferograms for workflow comparison. | [ARIA standard displacement products](https://aria.jpl.nasa.gov/products/standard-displacement-products.html) |
| MintPy | Time-series analysis and visualization layer for InSAR products. | [MintPy GitHub](https://github.com/insarlab/MintPy) |
| GNSS | Independent validation of displacement time series and velocities. | [Nevada Geodetic Laboratory](https://geodesy.unr.edu/index.php), [EarthScope GNSS data](https://www.earthscope.org/data/) |

The next notebook should choose one InSAR product path and document exact product versions, access date, spatial bounds, and preprocessing assumptions.


## Benchmark Workflow

```mermaid
flowchart LR
    A[Study area config] --> B[Open InSAR products]
    B --> C[Analysis-ready displacement time series]
    D[GNSS time series] --> E[Date and reference alignment]
    C --> E
    E --> F[Validation metrics]
    F --> G[Uncertainty and sensitivity checks]
    G --> H[Figures and interpretation]
```

The first benchmark should produce a small, defensible result before expanding to larger regions or multiple workflows.


## Success Criteria For The First Case Study

A first scientifically useful result should include:

1. a documented InSAR product source and time window,
2. a small set of GNSS stations inside or near the study box,
3. date-aligned InSAR and GNSS displacement time series,
4. validation metrics from `insar_benchmark_lab.metrics`,
5. at least one sensitivity test for reference area or masking,
6. figures that clearly distinguish measured results from assumptions.


## Next Notebook

Proceed to `02_hyp3_or_opera_to_timeseries.ipynb` after choosing the first data path. The recommended first path is OPERA DISP-S1 if a suitable product is available for the Central Valley bounds; otherwise use ASF HyP3 products and convert them into a small analysis-ready time-series example.
